<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/etapa-01-logica/07_Validade_e_Inferencia_Logica_na_Seguranca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/etapa-01-logica/07%20-%20Validade%20e%20Inferencia%20Logica%20na%20Seguranca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 07: Validade de Argumentos e Inferência Lógica na Segurança de Processos

**Projeto Integrador / Disciplina:** Matemática Discreta e Sistemas Digitais  
**Curso:** Engenharia de Controle e Automação (ECA)  
**Sistema:** Linha Automatizada de Envasamento e Tampamento de Bebidas  
**Grupo:** Grupo 3 — ECAA08  

---  
### Objetivo do Notebook
Implementar a classe **`ProvadorDedutivoFormal`** para verificar matematicamente a validade de argumentos dedutivos, regras de intertravamento de segurança (*Safety Interlocks*) e permissivos da **Linha de Envasamento de Bebidas (Grupo 3)**. Avaliaremos provas exaustivas por tabela-verdade ($2^n$ estados), verificação por refutação (*Reductio ad Absurdum* / SAT Solver) e identificação algorítmica de falácias industriais.

## 1. Núcleo Algorítmico: Provador Dedutivo Formal

Um argumento dedutivo $P_1, P_2, \dots, P_k \vdash C$ é válido se e somente se a conclusão $C$ for estritamente verdadeira em todas as valorações onde a conjunção das premissas $(P_1 \land P_2 \land \dots \land P_k)$ for verdadeira.

Pelo **Teorema da Dedução** e pela técnica de **Refutação (Reductio ad Absurdum)**:
$$\{P_1, P_2, \dots, P_k\} \models C \iff (P_1 \land P_2 \land \dots \land P_k) \rightarrow C \equiv \mathbf{T} \iff \{P_1, \dots, P_k, \neg C\} \equiv \mathbf{F}$$

In [ ]:
import itertools
from typing import List, Dict, Callable, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionarios em tabela ASCII pura padronizada."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)


class ProvadorDedutivoFormal:
    @staticmethod
    def verificar_argumento_tabela_verdade(
        variaveis: List[str],
        premissas: List[Callable[[Dict[str, bool]], bool]],
        conclusao: Callable[[Dict[str, bool]], bool]
    ) -> Dict[str, Any]:
        """
        Verifica a validade semantica do argumento: P1, P2, ..., Pk |- C por tabela-verdade exaustiva.
        Um argumento e valido sse em toda linha onde todas as premissas sao TRUE,
        a conclusao tambem for estritamente TRUE.
        """
        n = len(variaveis)
        total_estados = 2 ** n
        linhas_criticas = 0  # Linhas onde a conjuncao das premissas e verdadeira
        linhas_validas = 0    # Linhas criticas onde a conclusao tambem e verdadeira
        contraexemplos = []

        for combo in itertools.product([False, True], repeat=n):
            env = dict(zip(variaveis, combo))
            premissas_satisfeitas = all(p(env) for p in premissas)

            if premissas_satisfeitas:
                linhas_criticas += 1
                if conclusao(env):
                    linhas_validas += 1
                else:
                    contraexemplos.append(env)

        valido = (linhas_criticas > 0) and (linhas_criticas == linhas_validas)

        return {
            "Total Estados (2^n)": total_estados,
            "Estados com Premissas True": linhas_criticas,
            "Estados com Conclusão True": linhas_validas,
            "Válido": valido,
            "Resultado Semântico": "ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA)" if valido else "FALÁCIA / ARGUMENTO INVÁLIDO",
            "Contraexemplos": contraexemplos
        }

    @staticmethod
    def verificar_por_refutacao(
        variaveis: List[str],
        premissas: List[Callable[[Dict[str, bool]], bool]],
        conclusao: Callable[[Dict[str, bool]], bool]
    ) -> Dict[str, Any]:
        """
        Prova por Contradição / Refutação (SAT Solver approach):
        O argumento P1..Pk |- C e valido se e somente se o conjunto {P1, ..., Pk, NOT C}
        for estritamente INSATISFATÍVEL (CONTRADIÇÃO / BOTTOM).
        """
        n = len(variaveis)
        modelos_refutacao = []

        for combo in itertools.product([False, True], repeat=n):
            env = dict(zip(variaveis, combo))
            if all(p(env) for p in premissas) and not conclusao(env):
                modelos_refutacao.append(env)

        is_insatisfativel = (len(modelos_refutacao) == 0)
        return {
            "Satisfaz Negação": len(modelos_refutacao) > 0,
            "Refutação Bem-Sucedida": is_insatisfativel,
            "Modelos Encontrados": len(modelos_refutacao),
            "Conclusão": "PROVA POR CONTRADIÇÃO: ARGUMENTO VÁLIDO" if is_insatisfativel else "REFUTAÇÃO FALHOU: CONTRADIÇÃO NÃO ENCONTRADA"
        }

print("[OK] Módulo ProvadorDedutivoFormal inicializado com sucesso para a Linha de Envase (Grupo 3)!")

[OK] Módulo ProvadorDedutivoFormal inicializado com sucesso para a Linha de Envase (Grupo 3)!


## 2. Bateria de Testes Industriais da Linha de Envasamento (Grupo 3)

Testaremos formalmente os esquemas canônicos de inferência adaptados às tags do nosso projeto:
1. **Modus Ponens (MP):** Subpressão em TS1 ($p_{mín1}$) acarreta desligamento da bomba BC1 ($
eg y_{bomba}$).
2. **Modus Tollens (MT):** Válvula VS2 aberta ($y_{válv2}$) requer fluxo em SQ2 ($q_{flx2}$). Sem fluxo $\implies$ Válvula VS2 inoperante.
3. **Silogismo Hipotético (SH):** Sobrepressão em AS1 ($p_{máx2}$) bloqueia VS2, e bloqueio de VS2 para a esteira RC1 $\implies$ Sobrepressão em AS1 para a esteira RC1.
4. **Silogismo Disjuntivo (SD):** Alimentação por BC1 ($y_{bomba}$) ou Linha Reserva AS1 ($y_{aux}$). Falha em BC1 $\implies$ Ativa Linha Reserva AS1.
5. **Resolução Proposicional (RES):** Fusão de cláusulas para disparo de trip unificado.
6. **Dilema Construtivo (DC):** Atuação de contingência em alívio ou estrangulamento para sobrepressão ou excesso de vazão.
7. **Falácia da Afirmação do Consequente:** $(p_{mín1} \rightarrow \neg y_{bomba}) \land \neg y_{bomba} \not\vdash p_{mín1}$ (Rejeição formal).
8. **Falácia da Negação do Antecedente:** $(e_{stop} \rightarrow \neg y_{válv2}) \land \neg e_{stop} \not\vdash y_{válv2}$ (Rejeição formal).

In [ ]:
# ==============================================================================
# 1. MODUS PONENS (MP): Subpressão de Sucção em TS1 -> Desarme da Bomba BC1
# ==============================================================================
vars_mp = ['p_min1', 'y_bomba']
p1_mp = lambda env: (not env['p_min1']) or (not env['y_bomba'])  # p_min1 -> not y_bomba
p2_mp = lambda env: env['p_min1']                               # Fato: Ocorreu p_min1
c_mp  = lambda env: not env['y_bomba']                          # Conclusao: not y_bomba

res_mp = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_mp, [p1_mp, p2_mp], c_mp)
ref_mp = ProvadorDedutivoFormal.verificar_por_refutacao(vars_mp, [p1_mp, p2_mp], c_mp)

# ==============================================================================
# 2. MODUS TOLLENS (MT): Válvula VS2 aberta -> Fluxo em SQ2. Sem fluxo -> VS2 inoperante
# ==============================================================================
vars_mt = ['y_valv2', 'q_flx2']
p1_mt = lambda env: (not env['y_valv2']) or env['q_flx2']  # y_valv2 -> q_flx2
p2_mt = lambda env: not env['q_flx2']                     # not q_flx2
c_mt  = lambda env: not env['y_valv2']                     # Conclusao: not y_valv2

res_mt = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_mt, [p1_mt, p2_mt], c_mt)
ref_mt = ProvadorDedutivoFormal.verificar_por_refutacao(vars_mt, [p1_mt, p2_mt], c_mt)

# ==============================================================================
# 3. SILOGISMO HIPOTÉTICO (SH): Sobrepressão AS1 -> Bloqueia VS2 -> Para Esteira RC1
# ==============================================================================
vars_sh = ['p_max2', 'y_valv2', 'cmd_rc1']
p1_sh = lambda env: (not env['p_max2']) or (not env['y_valv2'])       # p_max2 -> not y_valv2
p2_sh = lambda env: env['y_valv2'] or (not env['cmd_rc1'])             # not y_valv2 -> not cmd_rc1
c_sh  = lambda env: (not env['p_max2']) or (not env['cmd_rc1'])       # Conclusao: p_max2 -> not cmd_rc1

res_sh = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_sh, [p1_sh, p2_sh], c_sh)
ref_sh = ProvadorDedutivoFormal.verificar_por_refutacao(vars_sh, [p1_sh, p2_sh], c_sh)

# ==============================================================================
# 4. SILOGISMO DISJUNTIVO (SD): Pressurização por BC1 ou Reserva AS1. Desarme BC1 -> Reserva AS1
# ==============================================================================
vars_sd = ['y_bomba', 'y_aux']
p1_sd = lambda env: env['y_bomba'] or env['y_aux']  # y_bomba or y_aux
p2_sd = lambda env: not env['y_bomba']              # not y_bomba
c_sd  = lambda env: env['y_aux']                   # Conclusao: y_aux

res_sd = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_sd, [p1_sd, p2_sd], c_sd)
ref_sd = ProvadorDedutivoFormal.verificar_por_refutacao(vars_sd, [p1_sd, p2_sd], c_sd)

# ==============================================================================
# 5. RESOLUÇÃO PROPOSICIONAL: Fusão de cláusulas de sobrepressão e emergência
# ==============================================================================
vars_res = ['p_max1', 'e_stop', 'trip_geral']
p1_res = lambda env: env['p_max1'] or env['e_stop']                  # p_max1 or e_stop
p2_res = lambda env: (not env['p_max1']) or env['trip_geral']        # not p_max1 or trip_geral
c_res  = lambda env: env['e_stop'] or env['trip_geral']              # Conclusao: e_stop or trip_geral

res_res = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_res, [p1_res, p2_res], c_res)
ref_res = ProvadorDedutivoFormal.verificar_por_refutacao(vars_res, [p1_res, p2_res], c_res)

# ==============================================================================
# 6. DILEMA CONSTRUTIVO (DC): Ações para sobrepressão SP1 ou sobrevazão SQ1
# ==============================================================================
vars_dc = ['p_max1', 'q_max1', 'y_psv', 'y_est']
p1_dc = lambda env: ((not env['p_max1']) or env['y_psv']) and ((not env['q_max1']) or env['y_est'])
p2_dc = lambda env: env['p_max1'] or env['q_max1']
c_dc  = lambda env: env['y_psv'] or env['y_est']

res_dc = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_dc, [p1_dc, p2_dc], c_dc)
ref_dc = ProvadorDedutivoFormal.verificar_por_refutacao(vars_dc, [p1_dc, p2_dc], c_dc)

# ==============================================================================
# 7. FALÁCIA: AFIRMAÇÃO DO CONSEQUENTE (INVÁLIDO)
# ==============================================================================
vars_fal_ac = ['p_min1', 'y_bomba']
p1_fal_ac = lambda env: (not env['p_min1']) or (not env['y_bomba'])
p2_fal_ac = lambda env: not env['y_bomba']
c_fal_ac  = lambda env: env['p_min1']  # Falácia: afirmar que a bomba está desligada implica subpressão

res_fal_ac = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_fal_ac, [p1_fal_ac, p2_fal_ac], c_fal_ac)
ref_fal_ac = ProvadorDedutivoFormal.verificar_por_refutacao(vars_fal_ac, [p1_fal_ac, p2_fal_ac], c_fal_ac)

# ==============================================================================
# 8. FALÁCIA: NEGAÇÃO DO ANTECEDENTE (INVÁLIDO)
# ==============================================================================
vars_fal_na = ['e_stop', 'y_valv2']
p1_fal_na = lambda env: (not env['e_stop']) or (not env['y_valv2'])
p2_fal_na = lambda env: not env['e_stop']
c_fal_na  = lambda env: env['y_valv2']  # Falácia: não estar em emergência implica válvula aberta

res_fal_na = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_fal_na, [p1_fal_na, p2_fal_na], c_fal_na)
ref_fal_na = ProvadorDedutivoFormal.verificar_por_refutacao(vars_fal_na, [p1_fal_na, p2_fal_na], c_fal_na)

# ==============================================================================
# RELATÓRIO CONSOLIDADO
# ==============================================================================
relatorio_testes = [
    {"Esquema Lógico": "Modus Ponens (MP)", "Variáveis": "p_min1, y_bomba", "Resultado Semântico": res_mp["Resultado Semântico"], "Válido": res_mp["Válido"], "Refutação": ref_mp["Refutação Bem-Sucedida"]},
    {"Esquema Lógico": "Modus Tollens (MT)", "Variáveis": "y_valv2, q_flx2", "Resultado Semântico": res_mt["Resultado Semântico"], "Válido": res_mt["Válido"], "Refutação": ref_mt["Refutação Bem-Sucedida"]},
    {"Esquema Lógico": "Silogismo Hipotético (SH)", "Variáveis": "p_max2, y_valv2, cmd_rc1", "Resultado Semântico": res_sh["Resultado Semântico"], "Válido": res_sh["Válido"], "Refutação": ref_sh["Refutação Bem-Sucedida"]},
    {"Esquema Lógico": "Silogismo Disjuntivo (SD)", "Variáveis": "y_bomba, y_aux", "Resultado Semântico": res_sd["Resultado Semântico"], "Válido": res_sd["Válido"], "Refutação": ref_sd["Refutação Bem-Sucedida"]},
    {"Esquema Lógico": "Resolução Proposicional (RES)", "Variáveis": "p_max1, e_stop, trip_geral", "Resultado Semântico": res_res["Resultado Semântico"], "Válido": res_res["Válido"], "Refutação": ref_res["Refutação Bem-Sucedida"]},
    {"Esquema Lógico": "Dilema Construtivo (DC)", "Variáveis": "p_max1, q_max1, y_psv, y_est", "Resultado Semântico": res_dc["Resultado Semântico"], "Válido": res_dc["Válido"], "Refutação": ref_dc["Refutação Bem-Sucedida"]},
    {"Esquema Lógico": "Afirmação Consequente (Falácia)", "Variáveis": "p_min1, y_bomba", "Resultado Semântico": res_fal_ac["Resultado Semântico"], "Válido": res_fal_ac["Válido"], "Refutação": ref_fal_ac["Refutação Bem-Sucedida"]},
    {"Esquema Lógico": "Negação Antecedente (Falácia)", "Variáveis": "e_stop, y_valv2", "Resultado Semântico": res_fal_na["Resultado Semântico"], "Válido": res_fal_na["Válido"], "Refutação": ref_fal_na["Refutação Bem-Sucedida"]}
]

print("=== RELATÓRIO DE VERIFICAÇÃO FORMAL DE ARGUMENTOS - GRUPO 3 (LINHA DE ENVASE) ===")
print(formatar_tabela(relatorio_testes))

# Asserções de integridade e segurança
assert res_mp["Válido"] is True and ref_mp["Refutação Bem-Sucedida"] is True
assert res_mt["Válido"] is True and ref_mt["Refutação Bem-Sucedida"] is True
assert res_sh["Válido"] is True and ref_sh["Refutação Bem-Sucedida"] is True
assert res_sd["Válido"] is True and ref_sd["Refutação Bem-Sucedida"] is True
assert res_res["Válido"] is True and ref_res["Refutação Bem-Sucedida"] is True
assert res_dc["Válido"] is True and ref_dc["Refutação Bem-Sucedida"] is True
assert res_fal_ac["Válido"] is False and ref_fal_ac["Refutação Bem-Sucedida"] is False
assert res_fal_na["Válido"] is False and ref_fal_na["Refutação Bem-Sucedida"] is False

print("\n[OK] 100% dos testes formais de inferencia e deteccao de falacias executados com exito!")

=== RELATÓRIO DE VERIFICAÇÃO FORMAL DE ARGUMENTOS - GRUPO 3 (LINHA DE ENVASE) ===
Esquema Lógico                  | Variáveis                    | Resultado Semântico                     | Válido | Refutação
--------------------------------+------------------------------+-----------------------------------------+--------+----------
Modus Ponens (MP)               | p_min1, y_bomba              | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True   | True     
Modus Tollens (MT)              | y_valv2, q_flx2              | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True   | True     
Silogismo Hipotético (SH)       | p_max2, y_valv2, cmd_rc1     | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True   | True     
Silogismo Disjuntivo (SD)       | y_bomba, y_aux               | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True   | True     
Resolução Proposicional (RES)   | p_max1, e_stop, trip_geral   | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True   | True     
Dilema Construtivo (DC)         | p_

## 3. Aplicação em Tempo Real: Motor de Diagnóstico e Intertravamento no SCADA

Para demonstrar o uso prático do provador dedutivo na rotina de varredura (*scan cycle*) do CLP da Linha de Envasamento, simulamos três quadros de telemetria da instrumentação com aplicação das inferências dedutivas de segurança.

In [ ]:
def diagnostico_scada(telemetria: Dict[str, Any]):
    """
    Executa inferências dedutivas sobre o vetor de telemetria recebido dos sensores.
    """
    sp1 = telemetria.get('SP1_Barg', 2.0)
    sq1 = telemetria.get('SQ1_Lmin', 25.0)
    sp2 = telemetria.get('SP2_Barg', 3.5)
    sq2 = telemetria.get('SQ2_Lmin', 5.0)
    e_stop = telemetria.get('E_STOP', False)
    cmd_bomba = telemetria.get('CMD_BC1', True)
    cmd_valv2 = telemetria.get('CMD_VS2', True)

    # Mapeamento proposicional
    p_min1 = sp1 < 1.0
    p_max1 = sp1 > 3.5
    p_max2 = sp2 > 4.5
    q_flx2 = sq2 >= 2.0

    acoes = []

    # Inferência 1: Modus Ponens para Subpressão na Sucção
    if p_min1:
        acoes.append("[TRIP MP] Subpressao em TS1 detectada (SP1 < 1.0 Barg) -> DESARMAR BOMBA BC1 IMEDIATAMENTE (Evitar Cavitação)!")

    # Inferência 2: Modus Tollens para Falha de Abertura da Válvula de Envase
    if cmd_valv2 and not q_flx2:
        acoes.append("[ALARME MT] Comando VS2 ativo mas SQ2 sem fluxo -> FALHA MECÂNICA / BOBINA QUEIMADA EM VS2!")

    # Inferência 3: Silogismo Hipotético para Sobrepressão no Acumulador
    if p_max2:
        acoes.append("[TRIP SH] Sobrepressao em AS1 (SP2 > 4.5 Barg) -> BLOQUEAR VS2 E PARAR ESTEIRA RC1!")

    if not acoes:
        acoes.append("[STATUS OK] Todos os parametros dentro dos limites nominais de seguranca.")

    return acoes

# Cenários de Teste
cenario_1 = {'SP1_Barg': 0.7, 'SQ1_Lmin': 10.0, 'SP2_Barg': 3.5, 'SQ2_Lmin': 5.0, 'CMD_BC1': True, 'CMD_VS2': False}
cenario_2 = {'SP1_Barg': 2.5, 'SQ1_Lmin': 20.0, 'SP2_Barg': 3.5, 'SQ2_Lmin': 0.0, 'CMD_BC1': True, 'CMD_VS2': True}
cenario_3 = {'SP1_Barg': 2.2, 'SQ1_Lmin': 25.0, 'SP2_Barg': 4.8, 'SQ2_Lmin': 6.0, 'CMD_BC1': True, 'CMD_VS2': True}

print("=== DIAGNÓSTICO EM TEMPO REAL SCADA (LINHA DE ENVASE) ===")
print("\n--- CENÁRIO 1 (Subpressão na Sucção TS1) ---")
for act in diagnostico_scada(cenario_1): print(" ", act)

print("\n--- CENÁRIO 2 (Comando VS2 Ativo Sem Vazão) ---")
for act in diagnostico_scada(cenario_2): print(" ", act)

print("\n--- CENÁRIO 3 (Sobrepressão no Acumulador AS1) ---")
for act in diagnostico_scada(cenario_3): print(" ", act)


=== DIAGNÓSTICO EM TEMPO REAL SCADA (LINHA DE ENVASE) ===

--- CENÁRIO 1 (Subpressão na Sucção TS1) ---
  [TRIP MP] Subpressao em TS1 detectada (SP1 < 1.0 Barg) -> DESARMAR BOMBA BC1 IMEDIATAMENTE (Evitar Cavitação)!

--- CENÁRIO 2 (Comando VS2 Ativo Sem Vazão) ---
  [ALARME MT] Comando VS2 ativo mas SQ2 sem fluxo -> FALHA MECÂNICA / BOBINA QUEIMADA EM VS2!

--- CENÁRIO 3 (Sobrepressão no Acumulador AS1) ---
  [TRIP SH] Sobrepressao em AS1 (SP2 > 4.5 Barg) -> BLOQUEAR VS2 E PARAR ESTEIRA RC1!
